# 配套实践 16-02：比较开环、MPC 与风险感知 MPC

本练习继续使用二维点机器人，但把解析 World Model 与真实系统故意设为不同。模型认为动作会被准确执行；真实系统在训练覆盖不足的中央区域受到向下扰动和更大噪声。我们比较一次规划后全部执行、每步重新规划，以及在规划代价中加入模型不确定性惩罚的三种控制方式。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/16-objective-cem-and-mpc/" target="_blank">在新标签页返回课程正文</a>


In [ ]:
import numpy as np  # 批量计算候选未来并重复随机试验
import matplotlib.pyplot as plt  # 绘制风险区域、闭环轨迹和统计指标
from matplotlib.patches import Circle  # 显示圆形障碍和安全边界
np.random.seed(21)  # 固定 NumPy 全局随机种子以便复现实验
plt.rcParams["figure.dpi"] = 120  # 提高笔记本内图像的显示清晰度
print("本练习使用轻量解析 World Model，完整统计通常可在数秒内完成。")  # 告知读者实验规模和硬件要求


## 1. World Model 不知道中央区域的真实扰动

机器人从左侧前往右侧目标。圆形障碍位于直线路径下方，模型中的直行路线看似有足够间隙；但中央区域缺少训练数据，真实系统会受到向下漂移和更强噪声。这里用一张连续风险图模拟第 15 章 ensemble 给出的不确定性，而不是假定规划器知道真实扰动方向。


In [ ]:
start = np.array([-1.20, 0.0])  # 设置所有控制器共享的初始位置
goal = np.array([1.20, 0.0])  # 设置所有控制器共享的目标位置
obstacle_center = np.array([0.0, -0.35])  # 把障碍放在直线路径下方
obstacle_radius = 0.31  # 设置真实碰撞判定使用的障碍半径
action_limit = 0.18  # 限制每个坐标上的单步动作幅度
maximum_steps = 28  # 设置真实闭环最多允许执行的步数
goal_tolerance = 0.18  # 设置判定任务成功的目标距离阈值
def uncertainty_level(positions):  # 定义训练覆盖不足区域的不确定性代理
    normalized_x = positions[..., 0] / 0.42  # 按中央区域横向尺度归一化位置
    normalized_y = positions[..., 1] / 0.32  # 按中央区域纵向尺度归一化位置
    return np.exp(-(normalized_x ** 2 + normalized_y ** 2))  # 用中心高、外围低的连续函数返回风险水平
x_grid = np.linspace(-1.45, 1.45, 260)  # 建立风险背景图的横向坐标
y_grid = np.linspace(-0.95, 0.95, 190)  # 建立风险背景图的纵向坐标
mesh_x, mesh_y = np.meshgrid(x_grid, y_grid)  # 组合二维绘图网格
risk_grid = uncertainty_level(np.stack([mesh_x, mesh_y], axis=-1))  # 计算每个网格位置的不确定性
figure, ax = plt.subplots(figsize=(8.0, 4.8), constrained_layout=True)  # 建立任务风险地图
risk_image = ax.contourf(mesh_x, mesh_y, risk_grid, levels=20, cmap="YlOrBr", alpha=0.55)  # 用连续底色显示风险代理
figure.colorbar(risk_image, ax=ax, label="Model uncertainty proxy")  # 说明底色代表模型不确定性而非代价
ax.add_patch(Circle(obstacle_center, obstacle_radius, color="#b94b55", alpha=0.55, label="Obstacle"))  # 绘制真实不可进入障碍
ax.plot([start[0], goal[0]], [start[1], goal[1]], color="#66788a", linestyle="--", linewidth=1.8, label="Nominal shortcut")  # 显示模型偏好的直接捷径
ax.scatter(*start, s=80, color="#245b78", zorder=5, label="Start")  # 标出任务起点
ax.scatter(*goal, s=125, marker="*", color="#2f855a", zorder=5, label="Goal")  # 标出任务目标
ax.set(xlabel="x position", ylabel="y position", title="The nominal shortcut crosses an uncertain region", xlim=(-1.45, 1.45), ylim=(-0.95, 0.95), aspect="equal")  # 设置任务图的坐标和范围
ax.legend(fontsize=8, loc="upper left")  # 显示任务场景图例
plt.show()  # 在笔记本中输出风险地图


**怎样理解结果：** 风险底色不是障碍，也没有告诉规划器真实扰动会向下。普通规划器只避开红色圆形障碍，仍可能选择穿过中央高不确定区域；风险感知规划器会为该区域付出附加代价，从而保留更大余量。


## 2. 同一个 CEM，通过风险权重改变候选排序

规划用的 World Model 仍是 $p_{t+1}=p_t+a_t$，不会模拟真实漂移。评价器包含目标、碰撞、缓冲、动作和途中距离；风险感知版本再累加 $\lambda_u U(p)$。下面先在同一初始状态分别规划一次，观察候选路径的区别。


In [ ]:
def predict_positions(initial_state, action_sequences):  # 定义规划器使用的批量解析 World Model
    return initial_state[None, None, :] + np.cumsum(action_sequences, axis=1)  # 累计动作得到全部候选的未来位置
def planning_cost(initial_state, action_sequences, risk_weight):  # 定义普通与风险感知规划共享的评价器
    predicted_positions = predict_positions(initial_state, action_sequences)  # 用名义 World Model 预测候选未来
    obstacle_distances = np.linalg.norm(predicted_positions - obstacle_center, axis=2)  # 计算预测轨迹到障碍中心的距离
    terminal_cost = 50.0 * np.sum((predicted_positions[:, -1] - goal) ** 2, axis=1)  # 强调 horizon 末端应接近目标
    progress_cost = 0.05 * np.sum(np.linalg.norm(predicted_positions - goal, axis=2), axis=1)  # 轻微鼓励沿途持续接近目标
    collision_cost = 500.0 * np.sum(obstacle_distances < obstacle_radius, axis=1)  # 大幅惩罚名义模型预测的碰撞
    clearance_cost = 10.0 * np.sum(np.maximum(obstacle_radius + 0.09 - obstacle_distances, 0.0) ** 2, axis=1)  # 在障碍外建立连续缓冲代价
    effort_cost = 0.01 * np.sum(action_sequences ** 2, axis=(1, 2))  # 轻微限制过大的控制动作
    uncertainty_cost = risk_weight * np.sum(uncertainty_level(predicted_positions), axis=1)  # 按候选经过的陌生区域累计风险代价
    return terminal_cost + progress_cost + collision_cost + clearance_cost + effort_cost + uncertainty_cost  # 返回用于 CEM 排序的总代价
def cem_plan(initial_state, plan_horizon, risk_weight, random_seed, initial_mean=None, sample_count=220, iteration_count=4):  # 定义可用于开环和 MPC 的 CEM 规划器
    random_generator = np.random.default_rng(random_seed)  # 为当前规划调用建立独立随机数生成器
    direction_to_goal = goal - initial_state  # 计算当前状态到目标的方向
    proposal_length = min(0.17, np.linalg.norm(direction_to_goal) / plan_horizon)  # 根据剩余距离限制初始 proposal 步长
    proposal_action = direction_to_goal / max(np.linalg.norm(direction_to_goal), 1e-6) * proposal_length  # 生成指向目标的基础动作 proposal
    action_mean = np.tile(proposal_action, (plan_horizon, 1)) if initial_mean is None else initial_mean.copy()  # 使用目标方向或上一计划作为均值初值
    action_std = np.full((plan_horizon, 2), 0.11)  # 用较宽标准差允许候选向障碍两侧探索
    global_best = None  # 保存当前规划调用中的全局最佳序列
    global_best_cost = np.inf  # 用无穷大初始化全局最佳代价
    final_sequences = None  # 预留最后一轮候选序列供可视化
    final_costs = None  # 预留最后一轮候选代价供可视化
    for iteration_index in range(iteration_count):  # 重复执行 CEM 候选采样和分布更新
        sampled_sequences = action_mean[None, :, :] + action_std[None, :, :] * random_generator.standard_normal((sample_count, plan_horizon, 2))  # 从当前动作分布采样候选序列
        sampled_sequences = np.clip(sampled_sequences, -action_limit, action_limit)  # 把候选动作裁剪到执行器边界
        sampled_costs = planning_cost(initial_state, sampled_sequences, risk_weight)  # 使用名义 World Model 评价全部候选
        elite_count = max(20, sample_count // 10)  # 取当前候选中成本最低的约十分之一
        elite_indices = np.argsort(sampled_costs)[:elite_count]  # 得到 elite 候选索引
        elite_sequences = sampled_sequences[elite_indices]  # 读取 elite 动作序列
        current_best_index = int(elite_indices[0])  # 读取当前轮最低代价候选索引
        if sampled_costs[current_best_index] < global_best_cost:  # 仅在确有改善时更新全局最佳结果
            global_best_cost = float(sampled_costs[current_best_index])  # 保存新的全局最低代价
            global_best = sampled_sequences[current_best_index].copy()  # 保存新的全局最佳动作序列
        action_mean = 0.15 * action_mean + 0.85 * elite_sequences.mean(axis=0)  # 使用 elite 平滑更新采样均值
        action_std = np.maximum(0.02, 0.15 * action_std + 0.85 * elite_sequences.std(axis=0))  # 更新采样标准差并保留最小探索宽度
        final_sequences = sampled_sequences  # 保存当前候选并在末轮后供教学图使用
        final_costs = sampled_costs  # 保存当前代价并在末轮后供教学图使用
    return global_best, action_mean, final_sequences, final_costs  # 返回最佳计划、warm start 均值和末轮候选
standard_plan, unused_standard_mean, standard_candidates, standard_costs = cem_plan(start, 18, 0.0, 31, sample_count=360, iteration_count=5)  # 在初始状态生成普通 CEM 计划
risk_plan, unused_risk_mean, risk_candidates, risk_costs = cem_plan(start, 18, 0.22, 31, sample_count=360, iteration_count=5)  # 用相同随机种子生成风险感知计划
figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), constrained_layout=True)  # 建立普通与风险感知候选对照图
planner_views = [(standard_candidates, standard_costs, standard_plan, "Standard cost"), (risk_candidates, risk_costs, risk_plan, "Risk-aware cost")]  # 组织两个规划器的绘图数据
for plot_index, (candidate_sequences, candidate_costs, best_sequence, plot_title) in enumerate(planner_views):  # 分别绘制两种代价下的候选
    best_candidate_indices = np.argsort(candidate_costs)[:20]  # 选取末轮代价最低的二十条候选
    candidate_paths = predict_positions(start, candidate_sequences[best_candidate_indices])  # 用名义 World Model得到低代价候选路径
    axes[plot_index].contourf(mesh_x, mesh_y, risk_grid, levels=20, cmap="YlOrBr", alpha=0.32)  # 在当前子图显示相同不确定性背景
    axes[plot_index].add_patch(Circle(obstacle_center, obstacle_radius, color="#b94b55", alpha=0.52))  # 绘制当前子图的障碍
    for candidate_path in candidate_paths:  # 逐条显示规划末轮的低代价候选
        complete_candidate = np.vstack([start, candidate_path])  # 为每条预测轨迹补上共同起点
        axes[plot_index].plot(complete_candidate[:, 0], complete_candidate[:, 1], color="#7e9bb7", linewidth=0.8, alpha=0.28)  # 用浅线显示候选未来
    complete_best = np.vstack([start, predict_positions(start, best_sequence[None, :, :])[0]])  # 得到当前规划器的最佳完整路径
    axes[plot_index].plot(complete_best[:, 0], complete_best[:, 1], color="#245b78", linewidth=2.3, label="Best predicted path")  # 突出当前规划器的最佳预测
    axes[plot_index].scatter(*start, s=55, color="#245b78", zorder=5)  # 标出共同起点
    axes[plot_index].scatter(*goal, s=90, marker="*", color="#2f855a", zorder=5)  # 标出共同目标
    axes[plot_index].set(title=plot_title, xlim=(-1.45, 1.45), ylim=(-0.95, 0.95), aspect="equal")  # 使用相同范围便于比较路径余量
    axes[plot_index].grid(alpha=0.16)  # 添加浅色网格帮助观察候选差别
plt.show()  # 在笔记本中输出两种候选排序的差异


**怎样理解结果：** 两个规划器使用完全相同的 World Model、CEM 随机数和障碍代价。唯一区别是右图对模型不确定区域额外计分，因此其低代价候选倾向于保留更大余量。风险权重并不知道扰动方向，只表达“这里的模型预测较不可信”。


## 3. 执行后重新观测，才能形成 MPC

真实系统在中央陌生区域产生向下漂移，并让动作噪声随不确定性增大。开环控制器在开始时规划 28 步并全部执行；普通 MPC 和风险感知 MPC 每一步读取实际位置，重新规划 14 步，只执行第一步。上一轮均值向前平移后作为 warm start。


In [ ]:
def real_system_step(state, action, random_generator):  # 定义含未知扰动和随机噪声的真实系统
    local_uncertainty = float(uncertainty_level(state[None, :])[0])  # 读取当前位置对应的陌生程度
    hidden_drift = np.array([0.0, -0.09 * local_uncertainty])  # 在陌生区域施加名义模型没有表示的向下漂移
    noise_std = 0.008 + 0.028 * local_uncertainty  # 让执行噪声也随陌生程度增大
    execution_noise = noise_std * random_generator.standard_normal(2)  # 为当前执行步采样随机误差
    return state + action + hidden_drift + execution_noise  # 返回真实系统执行后的新位置
def run_controller(controller_name, random_seed):  # 定义一次完整的开环或 MPC 任务执行
    random_generator = np.random.default_rng(random_seed)  # 为真实系统建立可复现的扰动序列
    current_state = start.copy()  # 从共同起点初始化真实状态
    real_path = [current_state.copy()]  # 保存每一步重新观测到的真实位置
    collision_happened = False  # 初始化整个 episode 的碰撞标志
    warm_start_mean = None  # 初始化 MPC 的上一轮计划均值
    if controller_name == "Open loop":  # 仅对开环控制器在开始时规划一次
        open_actions, unused_mean, unused_candidates, unused_costs = cem_plan(current_state, maximum_steps, 0.0, 1000 + random_seed, sample_count=450, iteration_count=6)  # 生成完整开环动作序列
    for step_index in range(maximum_steps):  # 在最大执行预算内逐步与真实系统交互
        if controller_name == "Open loop":  # 开环控制器不读取新状态重新求解计划
            selected_action = open_actions[step_index]  # 直接执行初始计划中当前位置的动作
        else:  # 两种 MPC 都会根据最新真实观测重新规划
            risk_weight = 0.22 if controller_name == "Risk-aware MPC" else 0.0  # 仅为风险感知版本启用不确定性代价
            planned_actions, updated_mean, unused_candidates, unused_costs = cem_plan(current_state, 14, risk_weight, 1000 + random_seed * 31 + step_index, initial_mean=warm_start_mean)  # 从最新状态求解新的短 horizon 计划
            selected_action = planned_actions[0]  # MPC 只执行新计划的第一个动作
            warm_start_mean = np.vstack([updated_mean[1:], updated_mean[-1:]])  # 将剩余均值向前平移作为下一轮初值
        current_state = real_system_step(current_state, selected_action, random_generator)  # 把选中动作发送到含扰动真实系统
        real_path.append(current_state.copy())  # 保存执行后重新观测到的真实位置
        collision_happened = collision_happened or np.linalg.norm(current_state - obstacle_center) < obstacle_radius  # 更新 episode 碰撞标志
        if np.linalg.norm(current_state - goal) < goal_tolerance:  # 检查真实状态是否已经进入成功范围
            break  # 成功后停止继续发送动作
    final_distance = float(np.linalg.norm(current_state - goal))  # 计算 episode 结束时的真实目标误差
    success = final_distance < goal_tolerance  # 根据统一阈值判断任务是否成功
    return {"path": np.asarray(real_path), "collision": collision_happened, "distance": final_distance, "success": success, "steps": len(real_path) - 1}  # 返回完整真实执行记录
example_seed = 0  # 选择能清楚显示三种控制差异的固定扰动序列
controller_names = ["Open loop", "Standard MPC", "Risk-aware MPC"]  # 定义要比较的三种控制器
example_results = {name: run_controller(name, example_seed) for name in controller_names}  # 在完全相同的随机扰动下运行三种控制器
controller_colors = {"Open loop": "#9a6b43", "Standard MPC": "#3b6ea8", "Risk-aware MPC": "#2f855a"}  # 为三种真实轨迹指定固定颜色
figure, ax = plt.subplots(figsize=(8.2, 5.0), constrained_layout=True)  # 建立单个扰动 episode 的真实轨迹对照图
ax.contourf(mesh_x, mesh_y, risk_grid, levels=20, cmap="YlOrBr", alpha=0.28)  # 显示模型不确定性区域背景
ax.add_patch(Circle(obstacle_center, obstacle_radius, color="#b94b55", alpha=0.55, label="Obstacle"))  # 绘制真实碰撞区域
for controller_name in controller_names:  # 逐个绘制三种控制器的真实执行轨迹
    result = example_results[controller_name]  # 读取当前控制器的 episode 结果
    status_text = "collision" if result["collision"] else "safe"  # 为图例生成简短安全状态
    ax.plot(result["path"][:, 0], result["path"][:, 1], marker=".", linewidth=2.1, color=controller_colors[controller_name], label=f"{controller_name}: {status_text}")  # 显示每一步真实观测位置
ax.scatter(*start, s=80, color="#245b78", zorder=5, label="Start")  # 标出三条轨迹的共同起点
ax.scatter(*goal, s=125, marker="*", color="#2f855a", edgecolor="white", linewidth=0.7, zorder=6, label="Goal")  # 标出任务目标
ax.set(xlabel="x position", ylabel="y position", title="Only executed states are shown", xlim=(-1.45, 1.45), ylim=(-0.95, 0.95), aspect="equal")  # 标注真实闭环轨迹图
ax.legend(fontsize=8, loc="upper left")  # 显示控制器名称和安全结果
ax.grid(alpha=0.16)  # 添加浅色网格帮助读取真实偏移
plt.show()  # 在笔记本中输出三种真实执行轨迹
for controller_name in controller_names:  # 逐个打印示例 episode 的可核查结果
    result = example_results[controller_name]  # 读取当前控制器的统计字段
    print(f"{controller_name:14s} | success={result['success']} | collision={result['collision']} | final distance={result['distance']:.3f} | steps={result['steps']}")  # 输出成功、安全、精度与步数


**怎样理解结果：** 图中只有真实执行状态，不包含未执行的候选。开环轨迹无法根据偏移修正；普通 MPC 会持续把当前位置重新送入规划器，但仍可能在模型过度自信的捷径附近进入障碍；风险感知 MPC 提前留出余量。单个随机 episode 只能解释机制，不能支持总体结论。


## 4. 用多次随机试验同时看成功与安全

下面对每种控制器运行 30 个随机种子。比较使用同一初始状态、目标、真实扰动规律和 CEM 预算；风险感知 MPC 只多出一个不确定性代价项。我们同时统计成功率、碰撞率、最终距离和执行步数，避免把更保守直接等同于更优秀。


In [ ]:
trial_count = 30  # 设置每种控制器重复运行的随机试验数量
all_results = {name: [] for name in controller_names}  # 为三种控制器分别建立结果列表
for trial_seed in range(trial_count):  # 使用相同的一组随机种子比较所有控制器
    for controller_name in controller_names:  # 在当前随机条件下依次运行三种控制器
        all_results[controller_name].append(run_controller(controller_name, trial_seed))  # 保存当前 episode 的真实闭环结果
success_rates = [np.mean([result["success"] for result in all_results[name]]) for name in controller_names]  # 计算每种控制器的任务成功率
collision_rates = [np.mean([result["collision"] for result in all_results[name]]) for name in controller_names]  # 计算每种控制器的碰撞 episode 比例
mean_distances = [np.mean([result["distance"] for result in all_results[name]]) for name in controller_names]  # 计算每种控制器的平均最终目标误差
mean_steps = [np.mean([result["steps"] for result in all_results[name]]) for name in controller_names]  # 计算每种控制器的平均执行步数
short_labels = ["Open", "MPC", "Risk MPC"]  # 为统计图准备紧凑横轴标签
bar_colors = [controller_colors[name] for name in controller_names]  # 按轨迹图中的颜色组织柱状图颜色
figure, axes = plt.subplots(1, 3, figsize=(12.2, 3.9), constrained_layout=True)  # 建立成功安全、目标误差和步数三个统计子图
bar_width = 0.36  # 设置并列成功率和碰撞率柱宽
x_positions = np.arange(len(controller_names))  # 建立三种控制器的横轴位置
axes[0].bar(x_positions - bar_width / 2, success_rates, width=bar_width, color="#4f8a62", label="Success rate")  # 绘制每种控制器的成功比例
axes[0].bar(x_positions + bar_width / 2, collision_rates, width=bar_width, color="#bd5d59", label="Collision rate")  # 绘制每种控制器的碰撞比例
axes[0].set(xticks=x_positions, xticklabels=short_labels, ylabel="Fraction of trials", title="Task and safety outcomes", ylim=(0.0, 1.05))  # 标注成功与安全统计图
axes[0].legend(fontsize=8)  # 显示成功率与碰撞率图例
axes[1].bar(short_labels, mean_distances, color=bar_colors)  # 绘制平均最终目标误差
axes[1].set(ylabel="Final distance to goal", title="Lower is better")  # 标注目标精度统计图
axes[2].bar(short_labels, mean_steps, color=bar_colors)  # 绘制平均真实执行步数
axes[2].set(ylabel="Executed steps", title="Safety can cost time")  # 标注执行效率统计图
for ax in axes:  # 对三个统计子图使用一致网格样式
    ax.grid(axis="y", alpha=0.18)  # 添加浅色水平网格帮助读取柱高
plt.show()  # 在笔记本中输出多次真实试验统计
for result_index, controller_name in enumerate(controller_names):  # 逐个打印三种控制器的汇总指标
    print(f"{controller_name:14s} | success={success_rates[result_index]:.1%} | collision={collision_rates[result_index]:.1%} | distance={mean_distances[result_index]:.3f} | steps={mean_steps[result_index]:.1f}")  # 输出成功、安全、精度与效率的联合结果


**怎样理解结果：** 开环控制无法修正未建模扰动，因此碰撞率和最终误差都较高。普通 MPC 依靠真实反馈显著改善到达率；风险感知 MPC 进一步降低碰撞，但可能绕得更远、用时更长，成功率也不一定在每个有限预算下最高。风险权重表达的是任务取舍，而不是免费提升所有指标。

**本练习的结论：** MPC 的核心不是“规划一次更长的轨迹”，而是执行短前缀后用真实观测持续修正。风险代价只有在不确定性与真实误差相关时才有意义，且不能替代硬碰撞检查。评价闭环时必须同时报告成功、安全、误差、耗时和计算预算；只看模型内最优成本会掩盖真正的执行失败。
